# ML Phase Detection — 3D Ising Model

Train a CNN on 2D spin configuration snapshots to learn the phase boundary.

Reference: Carrasquilla & Melko, Nature Physics 13, 431 (2017)

- Input: N×N spin slice (z=0 layer of 3D cubic lattice)
- Label: 0 = ordered (T < Tc), 1 = disordered (T > Tc)
- The network learns Tc without being told explicitly

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Optional torch import — graceful fallback if not installed
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    print("torch not installed. Install with: pip install torch")

from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr

# Load snapshot data
snap_path = 'data/snapshots_N20.csv'
if not os.path.exists(snap_path):
    print(f"ERROR: {snap_path} not found.")
    print("Run: cargo run --release --bin sweep -- --n 20 --tmin 2.0 --tmax 7.0 --steps 50 --warmup 2000 --samples 500 --save-snapshots --outdir analysis/data")
    raise FileNotFoundError(snap_path)

df = pd.read_csv(snap_path)
T_c = 4.5115  # 3D Ising critical temperature (J/kB)
N = 20

print(f"Loaded: {len(df)} snapshots, {df.shape[1]} columns")
print(f"Temperature range: {df['temperature'].min():.2f} – {df['temperature'].max():.2f}")
print(f"Tc = {T_c}")

In [ ]:
X_raw = df.iloc[:, :-1].values.astype(np.float32)  # spin values ±1
T_vals = df['temperature'].values

# Map {-1, +1} -> {0, 1} for CNN (standard binary image convention)
X = (X_raw + 1) / 2
X_img = X.reshape(-1, 1, N, N)  # (samples, channels, H, W)

# Labels: 0=ordered (T<Tc), 1=disordered (T>Tc)
y = (T_vals > T_c).astype(np.int64)

print(f"X shape: {X_img.shape}")
print(f"Ordered (y=0): {(y==0).sum()}, Disordered (y=1): {(y==1).sum()}")
print(f"Class balance: {(y==0).mean()*100:.1f}% ordered")

In [ ]:
if TORCH_AVAILABLE:
    class IsingCNN(nn.Module):
        def __init__(self, n):
            super().__init__()
            self.conv = nn.Sequential(
                nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(),
                nn.MaxPool2d(2),                          # -> n/2
                nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(),
                nn.MaxPool2d(2),                          # -> n/4
            )
            flat = 32 * (n // 4) * (n // 4)
            self.fc = nn.Sequential(
                nn.Flatten(),
                nn.Linear(flat, 64), nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(64, 2),
            )

        def forward(self, x):
            return self.fc(self.conv(x))

    model = IsingCNN(N)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"IsingCNN: {n_params:,} parameters")
    print(model)

In [ ]:
if TORCH_AVAILABLE:
    X_train, X_test, y_train, y_test, T_train, T_test = train_test_split(
        X_img, y, T_vals, test_size=0.2, random_state=42, stratify=y
    )

    train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    test_ds  = TensorDataset(torch.tensor(X_test),  torch.tensor(y_test))
    train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
    test_dl  = DataLoader(test_ds,  batch_size=32)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    train_losses, test_accs = [], []
    n_epochs = 30

    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0.0
        for xb, yb in train_dl:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        train_losses.append(epoch_loss / len(train_dl))

        model.eval()
        with torch.no_grad():
            correct = sum((model(xb).argmax(1) == yb).sum().item() for xb, yb in test_dl)
        test_accs.append(correct / len(y_test))

        if (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1:2d}/{n_epochs}: loss={train_losses[-1]:.4f}, test_acc={test_accs[-1]*100:.1f}%")

    print(f"\nFinal test accuracy: {test_accs[-1]*100:.1f}%")

In [ ]:
if TORCH_AVAILABLE:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.plot(train_losses, color='#3b82f6')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Training loss')
    ax1.set_title('CNN training loss')
    ax1.grid(True, alpha=0.3)

    ax2.plot([a*100 for a in test_accs], color='#10b981')
    ax2.axhline(90, color='gray', linestyle=':', alpha=0.5, label='90% line')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Test accuracy (%)')
    ax2.set_title('CNN test accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('data/ml_training_curve.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
if TORCH_AVAILABLE:
    model.eval()
    X_all = torch.tensor(X_img)
    with torch.no_grad():
        logits = model(X_all)
        probs_ordered = torch.softmax(logits, dim=1)[:, 0].numpy()  # P(ordered)

    df_res = pd.DataFrame({'T': T_vals, 'P_ordered': probs_ordered})
    df_mean = df_res.groupby('T')['P_ordered'].mean().reset_index()

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(df_mean['T'], df_mean['P_ordered'], 'o-', color='#3b82f6', markersize=6)
    ax.axvline(T_c, color='red', linestyle='--', linewidth=1.5, label=f'Tc = {T_c}')
    ax.axhline(0.5, color='gray', linestyle=':', alpha=0.5, label='P=0.5')
    ax.set_xlabel('Temperature T (J/kB)', fontsize=12)
    ax.set_ylabel('P(ordered)', fontsize=12)
    ax.set_title('CNN learned phase boundary (3D Ising)', fontsize=13)
    ax.legend(fontsize=11)
    ax.set_ylim([-0.05, 1.05])
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('data/ml_phase_boundary.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Estimate Tc from network
    crossing = df_mean[df_mean['P_ordered'] <= 0.5]
    if len(crossing) > 0:
        tc_est = crossing.iloc[0]['T']
        print(f"Network Tc estimate: {tc_est:.3f}  (theory: {T_c})")
        print(f"Error: {abs(tc_est - T_c):.3f} J/kB ({abs(tc_est-T_c)/T_c*100:.1f}%)")
    else:
        print("P(ordered) never drops below 0.5 — T range may not extend far enough above Tc")

## PCA Baseline

PCA provides an unsupervised sanity check: if PC1 correlates strongly with |M| (magnetisation), 
the data has clear structure that a linear method can also detect.

In [ ]:
# PCA works regardless of torch availability
X_flat = X_raw  # raw ±1 spins (N*N features)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_flat)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

scatter = ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=T_vals, cmap='coolwarm', alpha=0.5, s=8)
plt.colorbar(scatter, ax=ax1, label='Temperature T (J/kB)')
ax1.set_xlabel('PC1', fontsize=11)
ax1.set_ylabel('PC2', fontsize=11)
ax1.set_title(f'PCA of spin configurations\n(PC1: {pca.explained_variance_ratio_[0]*100:.1f}%, PC2: {pca.explained_variance_ratio_[1]*100:.1f}%)', fontsize=11)

# PC1 vs temperature
ax2.scatter(T_vals, X_pca[:, 0], c=T_vals, cmap='coolwarm', alpha=0.4, s=8)
ax2.axvline(T_c, color='red', linestyle='--', label=f'Tc={T_c}')
ax2.set_xlabel('Temperature T (J/kB)', fontsize=11)
ax2.set_ylabel('PC1', fontsize=11)
ax2.set_title('PC1 vs Temperature', fontsize=11)
ax2.legend()

plt.tight_layout()
plt.savefig('data/ml_pca.png', dpi=150, bbox_inches='tight')
plt.show()

# Correlation with order parameter
M_vals = np.abs(X_flat.mean(axis=1))
r, p_val = pearsonr(X_pca[:, 0], M_vals)
print(f"PC1 explains {pca.explained_variance_ratio_[0]*100:.1f}% of variance")
print(f"Pearson r(PC1, |M|) = {r:.3f}, p = {p_val:.2e}")
print(f"Expected: |r| > 0.9, confirming PC1 captures the order parameter")

## Summary

| Method | Tc estimate | Error |
|--------|-------------|-------|
| CNN    | see above   | —     |
| Theory | 4.5115      | 0%    |

Notes:
- CNN trained on N=20 slices with T range 2.0–7.0 J/kB, 50 temperatures, 10 snapshots each
- PCA PC1 should strongly correlate with |M| — this confirms the snapshot data has physical structure
- For publication: use larger N (40+), more temperatures, more snapshots per T
- Reference: Carrasquilla & Melko, Nature Physics 13, 431 (2017)